### Schema Validation
1. What is Schema-on-read / Schema-on-write?
2. Column Order Validation
3. Data Type Validation
4. Column Name Validation
5. Nullability Validation
6. Extra Columns Validation

### What is Schema-On-Read / Schema-On-Write?

---
These terms represent two fundamentally different approaches to how data systems handle structure, validation, and storage. 
- The easiest way to think about the difference is **when** the rules (the schema) are enforced: 
    - before the data lands on disk.
    - when you write a query to read it.

---
### 🗺️ The Concepts Visualized
---

### 1. Schema-on-Write (The Gatekeeper Approach)

In a Schema-on-Write system, the database management system **enforces the structure upfront**. Before a single row of data is physically written to disk, the engine checks it against a strict, pre-defined blueprint (columns, data types, constraints).

* **How it works:** If your table expects an `INT` for `customer_id`, and a pipeline tries to insert the string `'ABC'`, the storage engine instantly rejects the write and throws an error.
* **Where it's used:** Traditional Data Warehouses (like Snowflake, Synapse) and **Delta Lake** (via Schema Enforcement).
* **Pros:** Absolute data quality. Downstream BI reports and analytics jobs will never crash due to corrupted or missing columns.
* **Cons:** Less flexible. If an upstream application adds a new column, your ingestion pipeline will break until you manually alter the table structure.

---

### 2. Schema-on-Read (The "Dump Now, Figure It Out Later" Approach)

In a Schema-on-Read system, the storage layer is completely indifferent to what the data looks like. You can dump structured, semi-structured (JSON), or completely unstructured data into a directory, and the storage layer will blindly accept it.

* **How it works:** The data just sits as raw files on disk. The "schema" is only applied when a developer or query engine reads the data by overlaying a structure on top of it at execution time.
* **Where it's used:** Traditional, raw **Data Lakes** (like raw Parquet, CSV, or JSON folders sitting on Azure ADLS Gen2 or AWS S3).
* **Pros:** Massive ingestion speed and total flexibility. Pipelines never break due to upstream format changes because the storage layer never blocks a write.
* **Cons:** It can easily turn a Data Lake into a **Data Swamp**. If a file format changes down the line, your downstream production models and dashboards will suddenly crash with type-mismatch exceptions.

---

### 📊 Quick-Reference Comparison

| Feature | Schema-on-Write (e.g., Delta Lake) | Schema-on-Read (e.g., Raw Parquet Lake) |
| --- | --- | --- |
| **Enforcement Time** | **During Ingestion** (Before writing to disk). | **During Querying** (When reading from disk). |
| **Data Quality** | **High & Guaranteed**. Clean and reliable. | **Unpredictable**. Risk of hidden corruption. |
| **Write Performance** | Slower (Compute spent validating schemas). | Faster (Blind data dumping). |
| **Flexibility** | Rigid (Requires Schema Evolution rules). | Highly flexible (Accepts any payload). |

---

### 💡 The Delta Lake Connection

This concept is exactly why layers like Delta Lake exist. A traditional data lake is purely **Schema-on-Read**. By mapping a transaction log over your Parquet files, **Delta Lake converts your data lake into a Schema-on-Write system**, giving you the cost benefits of a data lake with the strict quality guarantees of a database warehouse.

---

### 📊 Interview Flashcard

> **Q: Why does a raw Data Lake follow Schema-on-Read, and how does Delta Lake change that behavior?**
> * A raw Data Lake stores data in unmanaged object storage files, applying structure only when a query parses the files (*Schema-on-Read*). Delta Lake introduces a transactional metadata layer that validates incoming records against a strict schema blueprint *before* committing the write (*Schema-on-Write*), protecting the lakehouse from schema corruption.
---

### Detail Summary On Schema Enforcement & Validation in Delta Lake

#### 📌 Core Architectural Meaning

Delta Lake utilizes a **Schema-on-Write** paradigm. Before any write transaction (`INSERT`, `APPEND`, `MERGE`) is officially committed to disk, Delta compares the structure of the incoming data against the table's current master schema stored in the transaction log (`_delta_log/`). If the incoming structure violates the integrity rules, Delta kills the transaction instantly to prevent **Data Swamp** corruption.

---

#### 📋 The 5 Pillars of Schema Validation

##### 1. Column Order Validation (The Position Trap)

* **The Meaning:** In traditional SQL, standard `INSERT INTO` statements map data fields from left to right based strictly on **ordinal position**, completely ignoring column names.
* **The Rule:** If you shuffle the order of columns in a query but the data types still match (e.g., swapping `customer_id` and `quantity` because both are integers), Delta will allow the write, but it will cause **silent data corruption**.
* **The Guardrail:** To prevent this layout trap, always declare explicit target columns in your insert statements, or rely on `MERGE INTO` which matches columns securely **by name** rather than position.

##### 2. Data Type Validation (The Type Guardian)

* **The Meaning:** Ensures that the data type of every incoming record field strictly matches the data type defined in the table schema.
* **The Rule:** Delta will proactively attempt safe implicit casting (e.g., converting a valid numeric string like `'99499'` into an `INT`, or a date string into a `DATE`). However, if an upstream system passes raw garbage text (like `'ABC'`) into an integer column, Delta kills the write immediately with a casting exception to protect structural integrity.

##### 3. Column Name Validation (The Name Check)

* **The Meaning:** Validates that the structural labels of the incoming dataset align cleanly with the target destination attributes.
* **The Rule:** Similar to column order, a standard positional `INSERT` statement is blind to names and will let a column named `C_ID` write directly into a column defined as `customer_id` as long as it sits in the right index position. Conversely, a `MERGE` or a data frame write requires an absolute, explicit name alignment and will fail immediately if a name mismatch is detected.

##### 4. Nullability Validation (The Constraint Gate)

* **The Meaning:** Enforces strict compliance with `NOT NULL` data design constraints set during table creation.
* **The Rule:** If a critical business operational field (like `customer_id`) is defined as `NOT NULL`, Delta scans incoming records before writing. The moment a transaction attempts to pass a `NULL` value into that protected slot, Delta aborts the write operation, throwing a constraint violation error.

##### 5. Extra Columns Validation (The Payload Shield)

* **The Meaning:** Prevents rogue or unmapped attributes from entering a cleanly defined production table layout.
* **The Rule:** If an upstream system introduces a brand-new, unexpected column (e.g., adding `customer_type` to a schema that doesn't expect it), Delta blocks the write instantly with a `SchemaMismatchedException`. This acts as a protective shield, forcing the data engineer to consciously choose to evolve the schema using **Schema Evolution** parameters (`mergeSchema = true`) before the data is allowed to land.

---

#### 📊 Interview-Ready Summary

* Delta Lake implements **Schema Enforcement** at the storage tier to guarantee that no write operation can corrupt the table’s structural blueprint. It validates structural alignment across five dimensions: checking ordinal positioning, rejecting incompatible data types, blocking unmapped extra columns, tracking explicit column names during advanced merges, and throwing immediate exceptions upon `NOT NULL` constraint violations.


In [0]:
%sql
-- In prevous notebook(i.e NB1, NB2), we created delta table using parquet files directly with --(CTAS).
-- Here we are first manully creating/defining schema for the delta table and then inserting data into it.

CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_sv (
  customer_id INT NOT NULL,
  invoice_no STRING,
  quantity INT,
  price FLOAT,
  invoice_date DATE
);

-- Used parquet files has many columns but we are using the below ones only.
INSERT INTO delta_catalog.delta_db.invoices_sv
  SELECT
    customer_id,
    invoice_no,
    quantity,
    price,
    invoice_date
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_sv;

In [0]:
%sql 
-- DROP TABLE delta_catalog.delta_db.invoices_sv;

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv
LIMIT 5;

In [0]:
%sql
SELECT
  MIN(customer_id) AS min_customer_id,
  MAX(customer_id) As max_customer_id,
  count(*) AS total_rows
FROM
  delta_catalog.delta_db.invoices_sv;

### Scenario 1: Column Order Validation

In [0]:
%sql
-- This cell attempts to insert a row into the delta_catalog.delta_db.invoices_sv table.
-- However, the SELECT statement's column order does not match the target delta table column order.
-- The target table expects in this order: customer_id, invoice_no, quantity, price, invoice_date.
-- The select statement provides in this order: quantity, invoice_no, customer_id, price, invoice_date.
-- This may cause a schema mismatch or incorrect data insertion.
-- As in our example both columns are of same type, so no error is thrown but data is inserted incorrectly.

-- Main query of cell startes here
INSERT INTO delta_catalog.delta_db.invoices_sv
  SELECT
    quantity,
    invoice_no,
    customer_id,
    price,
    invoice_date
  FROM
    VALUES(9999, 'I12345', 10, 100, '2022-01-01') 
    AS T(customer_id, invoice_no, quantity, price, invoice_date);
-- Main query of cell ends here

-- Run only below code to see the select order of columns and types.
-- result of below query will show correct value for columns but its not how the actual data is inserted.
SELECT
    quantity,
    invoice_no,
    customer_id,
    price,
    invoice_date
  FROM
    VALUES(9999, 'I12345', 10, 100, '2022-01-01') 
    AS T(customer_id, invoice_no, quantity, price, invoice_date);

In [0]:
%sql
-- let's see if the data was inserted correctly with customer_id = 9999
-- Ans : No (result is empty for this query)
-- This is because the column order in the select statement does not match target table schema/column order.
-- The target table expects in this order: customer_id, invoice_no, quantity, price, invoice_date.
-- The select statement provides in this order: quantity, invoice_no, customer_id, price, invoice_date.
-- This has caused data corrupted.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv
WHERE customer_id = 9999;

In [0]:
%sql
-- Now, let's see if the data was inserted with customer_id = 10.
-- Ans : Yes (result contains two records with customer_id = 10).
-- As, We already had one row for customer_id = 10.
-- This happend because the data was inderted based on column matching by position and not by column name.
-- customer_id, invoice_no, quantity, price, invoice_date (Column order in delta table).
-- quantity, invoice_no, customer_id, price, invoice_date (Column order in select statement).
-- Thats why we see row inserted for customer_id = 10 and not for customer_id = 9999.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv
WHERE customer_id = 10;

**Note :- **
- In Apache Spark and Delta Lake, a standard INSERT INTO statement completely ignores column names in your SELECT clause. Instead, it maps data strictly by ordinal position (left-to-right).
- **Conclusion** : Only use INSERT INTO when we are sure about the schema and column order of the table we want to write to, Else we would end up with corrupted data as shown above.

In [0]:
%sql
SELECT
  customer_id,
  invoice_no,
  quantity,
  price,
  invoice_date
FROM
  PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`
ORDER BY
  customer_id
LIMIT 5;

**A MERGE INTO statement natively resolves columns by name rather than position, completely avoiding this positional mismatch trap like INSERT INTO does.**

In [0]:
%sql
-- Now, here also we have put different column order in the select statement.
-- Only the first 5 rows(101-105) from the source parquet file are being appended to the target table.
MERGE INTO
  delta_catalog.delta_db.invoices_sv AS target
USING (
  SELECT
    quantity,
    invoice_no,
    customer_id,
    CAST(price AS FLOAT),
    invoice_date
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY
    customer_id
  LIMIT 5
) AS source
ON
  target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET
  target.customer_id = source.customer_id,
  target.invoice_no = source.invoice_no,
  target.quantity = source.quantity,
  target.price = source.price,
  target.invoice_date = current_date
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv
WHERE customer_id > 100;

-- The output shows customer_id from 101 to 105 inserted correctly in the table.
-- Thats means MERGE statement inserted the data correctly.

In [0]:
%sql
SELECT
  MIN(customer_id) AS min_customer_id,
  MAX(customer_id) As max_customer_id,
  count(*) AS total_rows
FROM
  delta_catalog.delta_db.invoices_sv;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_sv;

### Scenario 2: Data Type Validation

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices_sv
  VALUES ('abc', 'I12345', 10, 100, '2022-01-01');

-- This Code will throw error, because first value should be int or else string containing integers(i.e '12345').

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices_sv
  VALUES ('12345', 'I12345', 10, 100, '2022-01-01');

-- This Works because delta tries to convert the provided value to cooresponding data type of the column(here int), -- So, here string '12345' can be casted to int and '2022-01-01' can be casted to date.
-- But if you try to insert a value that can't be converted to cooresponding data type of the column, it will throw an error, as shown in above cell.

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv
WHERE
  customer_id = 12345;

### Scenario 3: Column Name Validation

In [0]:
%sql
-- This works.
-- invoices_sv table schema has different columns name then select statement schema.
-- invoices_sv table schema = customer_id, invoice_no, quantity, price, invoice_date.
-- Again as discusssed above as well, INSERT INTO append data based on position of columns and does not check column name, resulting in data insertion irrespective of what column name is in select.
-- As long as data type of columns matches (Also if possible, tries to cast to appropriate data type), data will be inserted.

INSERT INTO delta_catalog.delta_db.invoices_sv
  SELECT
    customer_id AS c_id,
    invoice_no,
    quantity AS qty,
    price,
    invoice_date
  FROM
    VALUES(9999, 'I12345', 10, 100, '2022-01-01') 
    AS T(customer_id, invoice_no, quantity, price, invoice_date);

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv
WHERE
  customer_id = 9999;

In [0]:
%sql
SELECT
    customer_id AS c_id,
    invoice_no,
    quantity AS qty,
    CAST(price AS FLOAT),
    invoice_date
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY
    customer_id desc
  LIMIT 5

In [0]:
%sql
-- This code throws error.
-- invoices_s table schema has different columns name then select statement schema.
-- invoices_sv table schema = customer_id, invoice_no, quantity, price, invoice_date.
-- Again as discusssed above as well, A MERGE INTO statement appends data based on columns by name rather than position and as the column name itself renamed in select it won't able to match columns while inserting, thus the code throws error.

MERGE INTO
  delta_catalog.delta_db.invoices_sv AS target
USING (
  SELECT
    customer_id AS c_id,
    invoice_no,
    quantity AS qty,
    CAST(price AS FLOAT),
    invoice_date
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY
    customer_id desc
  LIMIT 5
) AS source
ON
  target.customer_id = source.c_id
WHEN MATCHED THEN UPDATE SET
  target.customer_id = source.c_id,
  target.invoice_no = source.invoice_no,
  target.quantity = source.qty,
  target.price = source.price,
  target.invoice_date = current_date
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
-- As expected, query results 0 records, as above merge into operation failed due to column name mismatch.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv
WHERE
  customer_id BETWEEN 196 AND 200;

### Scenario 4: Nullability Validation

In [0]:
%sql
-- Throws error, because customer_id is not nullable in the table schema and constraint is violated for column customer_id.

INSERT INTO delta_catalog.delta_db.invoices_sv
  VALUES (NULL,NULL, NULL, NULL,NULL);

In [0]:
%sql
-- This works because customer_id is the only column that has non nullable in the table schema.

INSERT INTO delta_catalog.delta_db.invoices_sv
  VALUES (79123, NULL, NULL, NULL, NULL);

In [0]:
%sql
-- returns 1 row as expected.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv
WHERE
  customer_id = 79123;

### Scenario 4: Extra Columns Validation

In [0]:
%sql
-- This Cell throws error, because customer_type is not a column in invoices_sv table schema.

INSERT INTO delta_catalog.delta_db.invoices_sv
  SELECT
    customer_id,
    invoice_no,
    quantity,
    price,
    invoice_date,
    "VIP" AS customer_type -- This column is not present in invoices_sv target table schema.
  FROM
    VALUES(9999, 'I12345', 10, 100, '2022-01-01') 
    AS T(customer_id, invoice_no, quantity, price, invoice_date);

In [0]:
%sql
SELECT
  customer_id,
  invoice_no,
  quantity,
  CAST(price AS FLOAT),
  invoice_date,
  "VIP" AS customer_type -- This column is not present in invoices_sv table schema.
FROM
  PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`
WHERE 
  customer_id BETWEEN 150 AND 155

In [0]:
%sql
-- Now, this cell doesn't throw error, but only inserts the data for columns present in invoices_sv table schema and ignores additional column customer_type.

MERGE INTO
  delta_catalog.delta_db.invoices_sv AS target
USING (
  SELECT
    customer_id,
    invoice_no,
    quantity,
    CAST(price AS FLOAT),
    invoice_date,
    "VIP" AS customer_type -- This column is not present in invoices_sv table schema.
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  WHERE
    customer_id BETWEEN 150 AND 155
) AS source
ON
  target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET
  target.customer_id = source.customer_id,
  target.invoice_no = source.invoice_no,
  target.quantity = source.quantity,
  target.price = source.price,
  target.invoice_date = current_date
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
-- returns inserted 6 row, but not the additonal column customer_type.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv
WHERE
  customer_id BETWEEN 150 AND 155;

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_sv;

SELECT
  MIN(customer_id) AS min_customer_id,
  MAX(customer_id) As max_customer_id,
  count(*) AS total_rows
FROM
  delta_catalog.delta_db.invoices_sv;